### 가설검정의 원리
- 눈에 보이는 차이가 우연인지, 진짜인지를 판단하는 방법
- 용어
    - 귀무가설, 대립가설
    - 유의수준
    - p-value

- 따릉이 데이터의 남녀 이용차이로 검정절차를 확인해보기

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px  

plt.rcParams["font.family"] = "Malgun Gothic"   # 한글 깨짐 방지(윈도우 기본 폰트)
plt.rcParams["axes.unicode_minus"] = False

In [3]:
df = pd.read_csv("../data/07-9_bike_ttareungi.csv")
df.head()

,대여일자,대여소번호,대여소명,대여구분코드,성별,연령대코드,이용건수,운동량,탄소량,이동거리(M),이용시간(분)
0,202606,1201,1201. 가락시장역 3번 출구,정기권,NaN,30대,123,8750.55,78.97,339960.75,2621
1,202606,5339,5339. 은빛1단지 아파트 앞,정기권,M,~10대,15,517.56,4.65,20107.79,173
2,202606,3928,3928.고척아이파크아파트 106동 앞,일일권,NaN,40대,7,444.12,4.00,17254.05,148
3,202606,102,102. 망원역 1번출구 앞,가족권(2시간),M,~10대,1,181.72,1.64,7060.00,66
4,202606,2265,2265. 이수고가차도 남단,일일권,F,50대,6,660.63,5.96,25665.78,220


#### 가설검정을 해야 할 때
- 데이터에서 두 그룹 값 달라보이기는 한데, 우연인지 진짜인지 구별할 필요 있을 때
- 내가 이야기하고자 하는 스토리에 중요한 영향을 미칠 요소일 때

### 1. 질문에서 시작
- "남성과 여성의 평균 이용 건수가 다를까?"

In [5]:
df_gender = df.dropna(subset="성별")
df_gender.head()

,대여일자,대여소번호,대여소명,대여구분코드,성별,연령대코드,이용건수,운동량,탄소량,이동거리(M),이용시간(분)
1,202606,5339,5339. 은빛1단지 아파트 앞,정기권,M,~10대,15,517.56,4.65,20107.79,173
3,202606,102,102. 망원역 1번출구 앞,가족권(2시간),M,~10대,1,181.72,1.64,7060.00,66
4,202606,2265,2265. 이수고가차도 남단,일일권,F,50대,6,660.63,5.96,25665.78,220
5,202606,5766,5766. 북위례1,정기권,M,20대,26,749.83,6.74,30385.49,153
6,202606,4091,4091. 방학역2번출구,정기권,M,30대,207,13492.29,121.57,524177.51,4372


In [12]:
male = df_gender[df_gender['성별']=='M']['이용건수']
female = df_gender[df_gender['성별']=='F']['이용건수']
male

1        15
3         1
5        26
6       207
7       116
       ... 
4988     68
4991     11
4995      1
4997     37
4999    170
Name: 이용건수, Length: 1682, dtype: int64

In [9]:
total_male = df_gender[df_gender['성별']=='M']['이용건수'].sum()
total_male

np.int64(83566)

In [10]:
total_female = df_gender[df_gender['성별']=='F']['이용건수'].sum()
total_female

np.int64(47247)

#### 이 차이가 유의미한 차이인가 아닌가?
- 귀무가설
    - 남녀의 전체 이용 건수가 같다
- 대립가설
    - 남녀의 전체 이용 건수가 다르다

#### 유의수준과 p-value
- p-value : 귀무가설이 맞다고 할 때, 현재 상황이 '우연히' 나올 확률
- 유의수준(보통 0.05) : 확률이 이 이하면, 우연이 아니다 -> 즉 대립가설이 채택된다는 기준선

In [14]:
from scipy import stats

# t 검정으로 p-value 구하기

t, p = stats.ttest_ind(male, female, equal_var=False)
print(t, "t 통계량")
print(f"p-value: {p:.4f}")

6.877884158343051 t 통계량
p-value: 0.0000


- p-value 값이 0.05보다 작으면 -> 차이가 없다 라는 귀무가설 기각!
- 즉, 대립가설 인용(채택)
- 즉, 남녀 차이는 우연이 아니라 통계적으로 유의미하다

#### 판단이 틀릴 수도 있다
- 1종 오류
    - 사실은 차이가 없는데 있다고 잘못 판단(억울한 판정)
- 2종 오류
    - 사실은 차이가 있는데 없다고 놓친 판단(놓친 판정)

- 유의수준이 0.05 -> 1종 오류를 5%까지는 감수하겠다는 약속.

#### t 검정
- 가장 자주 만나는 질문 : 두 그룹의 평균이 다른가? 란 질문
- 가장 기본적인 판단 기준
------
- 일표본 : 한 그룹의 평균이 특정 기준과 다른가?
    - 예) 평균 이용 건수가 70분과 다른가?
- 독립표본 : 서로 다른 두 그룹 비교
    - 예) 남성과 여성
- 대응표본 : 같은 대상의 두 조건 비교
    - 예) 같은 상권의 주중 vs 주말

#### 1. 일표본 t 검정 - 기준값과 다른가
- 따릉이 평균 이용 건수가 40건과 다른가?

In [31]:
t, p = stats.ttest_1samp(df_gender['이용건수'], 40) # 일표본은 ttest_1samp 사용

print("평균 이용 건수:", df_gender['이용건수'].mean())
print(f"p-value: {p:.4f}")

평균 이용 건수: 41.03293601003764
p-value: 0.4504


#### 같은 대상의 두 조건
- 상권들의 주중 매출과 주말 매출이 다를까?

#### 서로 다른 그룹 비교
- 남성과 여성의 평균 이용 시간/건별

In [33]:
df_gender['이용시간(분)/건별'] = round(df_gender['이용시간(분)'] / df_gender['이용건수'],2)
df_gender.head(3)

,대여일자,대여소번호,대여소명,대여구분코드,성별,연령대코드,이용건수,운동량,탄소량,이동거리(M),이용시간(분),이용시간(분)/건별
1,202606,5339,5339. 은빛1단지 아파트 앞,정기권,M,~10대,15,517.56,4.65,20107.79,173,11.53
3,202606,102,102. 망원역 1번출구 앞,가족권(2시간),M,~10대,1,181.72,1.64,7060.00,66,66.00
4,202606,2265,2265. 이수고가차도 남단,일일권,F,50대,6,660.63,5.96,25665.78,220,36.67


In [34]:
male_time = df_gender[df_gender['성별']=='M']['이용시간(분)/건별']
female_time = df_gender[df_gender['성별']=='F']['이용시간(분)/건별']

In [36]:
t, p = stats.ttest_ind(male_time, female_time, equal_var=False) # 독립표본은 ttest_ind 사용

print(f"남자 평균: {male_time.mean():.3f}")
print(f"여자 평균: {female_time.mean():.3f}")

print(f"p-value: {p:.4f}")

남자 평균: 29.972
여자 평균: 31.717
p-value: 0.1172
